# Decision curve plotting
- This code plots decision curves for the models that have already been trained

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
TESTSET_Y_TRUE_PATH = r""

RANDOM_FOREST_TESTSET_Y_PRED = r""
XGBOOST_TESTSET_Y_PRED = r""
MLP_TESTSET_Y_PRED = r""

RULEFIT_TESTSET_Y_PRED = r""
EBM_TESTSET_Y_PRED = r""
DECISIONTREE_TESTSET_Y_PRED = r""
BRL_TESTSET_Y_PRED= r""
LOGREG_TESTSET_Y_PRED = r""

In [ ]:
logreg_y_pred = np.load(LOGREG_TESTSET_Y_PRED)
brl_y_pred = np.load(BRL_TESTSET_Y_PRED)
decision_tree_y_pred = np.load(DECISIONTREE_TESTSET_Y_PRED)
rulefit_y_pred = np.load(RULEFIT_TESTSET_Y_PRED)
ebm_y_pred = np.load(EBM_TESTSET_Y_PRED)

random_forest_y_pred = np.load(RANDOM_FOREST_TESTSET_Y_PRED)
xgboost_y_pred = np.load(XGBOOST_TESTSET_Y_PRED)
mlp_y_pred = np.load(MLP_TESTSET_Y_PRED)

y_true = pd.read_pickle(TESTSET_Y_TRUE_PATH)


In [ ]:
# decision curve analysis
plt.style.use("ggplot")

def plot_decision_curve(y_true, y_prob, max_threshold=0.5, save=False, filename=None, dpi=600, model_name="model"):
    
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    thresholds = np.arange(0.001, max_threshold, 0.01)
    n = len(y_true)
    prevalence = np.mean(y_true)

    net_benefit = []
    treat_all = []

    for pt in thresholds:
        predicted_positive = y_prob >= pt

        tp = np.sum(predicted_positive & (y_true == 1))
        fp = np.sum(predicted_positive & (y_true == 0))

        nb = (tp / n) - (fp / n) * (pt / (1 - pt))
        net_benefit.append(nb)

        nb_all = prevalence - (1 - prevalence) * (pt / (1 - pt))
        treat_all.append(nb_all)

    fig, ax = plt.subplots(figsize=(6, 3))

    ax.grid(False)
    ax.set_facecolor("white")
    for spine in ["bottom", "left"]:
        ax.spines[spine].set_color("black")
        ax.spines[spine].set_linewidth(1)
    

    ax.plot(thresholds, treat_all, color="gray", linestyle="--", label="Treat all", lw=0.9)
    ax.plot(thresholds, np.zeros_like(thresholds), linestyle="-", color="black", label="Treat none", lw=0.9)
    ax.plot(thresholds, net_benefit, color="royalblue", label=model_name)
    
    ax.set_xticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticks([-0.2, -0.1, 0, 0.1, 0.2])
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.25, 0.25)
    ax.tick_params(axis="both", labelsize=8)

    ax.set_xlabel("Threshold probability", fontsize=9)
    ax.set_ylabel("Net benefit", fontsize=9)
    ax.legend(fontsize=8, facecolor="white", frameon=False)

    if save == False:
        plt.show()

    if save == True:
        plt.savefig(f"{filename}", dpi=dpi, bbox_inches="tight")

In [ ]:
plot_decision_curve(y_true, logreg_y_pred, max_threshold=1, model_name="Logistic regression", filename="logreg_30d_readmission_dca.tiff", save=True)

In [ ]:
plot_decision_curve(y_true, brl_y_pred, max_threshold=1, model_name="Bayesian rule list", filename="BRL_30d_readmission_dca.tiff", save=True)

In [ ]:
plot_decision_curve(y_true, decision_tree_y_pred, max_threshold=1, model_name="Decision tree", filename="decisiontree_30d_readmission_dca.tiff", save=True)

In [ ]:
plot_decision_curve(y_true, ebm_y_pred, max_threshold=1, model_name="Explainable boosting machine", filename="EBM_30d_readmission_dca.tiff", save=True)

In [ ]:
plot_decision_curve(y_true, rulefit_y_pred, max_threshold=1, model_name="RuleFit", filename="RuleFit_30d_readmission_dca.tiff", save=True)

In [ ]:
plot_decision_curve(y_true, random_forest_y_pred, max_threshold=1, model_name="Random forest", filename="randomforest_30d_readmission_dca.tiff", save=True)

In [ ]:
plot_decision_curve(y_true, xgboost_y_pred, max_threshold=1, model_name="XGBoost", filename="xgboost_30d_readmission_dca.tiff", save=True)

In [ ]:
plot_decision_curve(y_true, mlp_y_pred, max_threshold=1, model_name="Multi-layer perceptron", filename="mlp_30d_readmission_dca.tiff", save=True)

### Single combined plot

In [ ]:
y_probs = [decision_tree_y_pred, brl_y_pred, logreg_y_pred, rulefit_y_pred, ebm_y_pred, mlp_y_pred, xgboost_y_pred, random_forest_y_pred]
model_types = [
    "IIML (non-additive)", "_IIML (non-additive)", "IIML (additive)",
    "_IIML (additive)", "_IIML (additive)", "Black-box", "_Black-box", "_Black-box"]
line_colours =["royalblue", "royalblue", "royalblue", "royalblue", "royalblue", "black", "black", "black"]
line_styles = [":", ":", "-", "-", "-", "-", "-", "-"]
line_alphas = [1, 1, 1, 1, 1, 0.7, 0.7, 0.7]

In [ ]:
def plot_multiple_unlabelled_decision_curves(
    y_true,
    y_probs,
    model_types,
    max_threshold=1.0,
    step=0.001,
    line_colours=None,
    line_styles=None,
    line_alphas=None,
    save=False,
    filename="compiled_unlablled_decision_curve.tiff",
    dpi=300
):
    """
    Plot decision curves for multiple models on the same axes.

    Parameters
    ----------
    y_true: array
        Binary outcomes (0/1)

    y_probs: list of arrays
        Probabilistic predictions from each model.

    model_names: list of str
        Names of the models - must be in the same order as the y_probs.

    max_threshold: float
        Maximum threshold probability to plot up to.

    step: float
        Step size for threshold probabilities.

    line_colours: list of str, optional
        Line colours for each model. Must be same length as y_probs.

    line_styles: list of str, optional
        Line styles for each model. Must be same length as y_probs.

    line_alphas: list of float, optional
        Line alphas for each model.  Must be same length as y_probs.

    save: bool
        Whether to save the resulting figure.

    filename: str
        Output filename.

    dpi: int
        Resolution for saved figure.
    """

    y_true = np.asarray(y_true)

    if len(y_probs) != len(model_names):
        raise ValueError("y_probs and model_types must have the same length.")

    if line_colours is not None and len(line_colours) != len(y_probs):
        raise ValueError("line_colours must have the same length as y_probs.")

    if line_styles is not None and len(line_styles) != len(y_probs):
        raise ValueError("line_styles must have the same length as y_probs.")

    thresholds = np.arange(0.001, max_threshold, step)
    n = len(y_true)
    prevalence = np.mean(y_true)

    # Treat-all strategy
    treat_all = prevalence - (1 - prevalence) * (thresholds / (1 - thresholds))

    fig, ax = plt.subplots(figsize=(8, 4))

    # Styling
    ax.grid(False)
    ax.set_facecolor("white")

    for spine in ["bottom", "left"]:
        ax.spines[spine].set_color("black")
        ax.spines[spine].set_linewidth(1)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Reference strategies
    ax.plot(
        thresholds,
        treat_all,
        color="gray",
        linestyle="--",
        label="Treat all",
        lw=0.9
    )

    ax.plot(
        thresholds,
        np.zeros_like(thresholds),
        linestyle="-",
        color="gray",
        label="Treat none",
        lw=0.9
    )

    # Model decision curves
    for i, (y_prob, model_type) in enumerate(zip(y_probs, model_types)):
        y_prob = np.asarray(y_prob)

        colour = line_colours[i] if line_colours is not None else None
        linestyle = line_styles[i] if line_styles is not None else "-"
        alpha = line_alphas[i] if line_alphas is not None else 1

        net_benefit = []

        for pt in thresholds:
            predicted_positive = y_prob >= pt

            tp = np.sum(predicted_positive & (y_true == 1))
            fp = np.sum(predicted_positive & (y_true == 0))

            nb = (tp / n) - (fp / n) * (pt / (1 - pt))
            net_benefit.append(nb)

        ax.plot(
            thresholds,
            net_benefit,
            color=colour,
            linestyle=linestyle,
            alpha=alpha,
            label=model_type,
            lw=1
        )

    # Axes
    ax.set_xticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticks([ 0, 0.1, 0.2])
    ax.set_xlim(0, max_threshold)
    ax.set_ylim(-0.05, 0.25)

    ax.tick_params(axis="both", labelsize=8)

    ax.set_xlabel("Threshold probability", fontsize=9, color="black")
    ax.set_ylabel("Net benefit", fontsize=9, color="black")

    ax.legend(fontsize=8, facecolor="white", frameon=False)

    plt.tight_layout()

    if save:
        plt.savefig(filename, dpi=dpi, bbox_inches="tight")
    else:
        plt.show()

In [ ]:
plot_multiple_unlabelled_decision_curves(
    y_true=y_true,
    y_probs=y_probs,
    model_types=model_types,
    line_colours=line_colours,
    line_styles=line_styles,
    line_alphas=line_alphas,
    max_threshold=1.0,
    step=0.001,
    save=True,
    filename="simplified_compiled_dca.tiff",
    dpi=600
)